# 🎬 AI Clone Studio — Hallo + BiRefNet (Pipeline 100% Gratuito)
### Criado para Aías Fernando (`@aiasfernando`) | Business do MMA / Gameplan

> **Status:** 100% Custo Zero via Google Colab (GPU NVIDIA T4/V100/A100 Gratuita)
>
> **Instruções Rápidas:**
> 1. No menu superior do Google Colab, vá em **Ambiente de Execução > Alterar tipo de ambiente de execução** e certifique-se de que **T4 GPU** está selecionada.
> 2. Clique no menu **Ambiente de Execução > Executar tudo** (ou execute as células em sequência).
> 3. Ao final da última célula, você receberá um link público `.gradio.live` para abrir a interface completa no seu navegador ou colar no seu estúdio local!

## 1. Verificação de GPU NVIDIA e CUDA

In [ ]:
!nvidia-smi

## 2. Instalação de Dependências e Clones dos Repositórios

In [ ]:
# Instalar pacotes de sistema e bibliotecas de ponta
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q --upgrade pip
!pip install -q diffusers transformers accelerate insightface mediapipe audio-separator timm einops kornia opencv-python soundfile torchaudio gradio pyyaml torchvision

# Clonar os repositórios oficiais
!rm -rf hallo birefnet
!git clone --depth 1 https://github.com/fudan-generative-vision/hallo.git
!git clone --depth 1 https://github.com/ZhengPeng7/BiRefNet.git

# Instalar hallo em modo de desenvolvimento
%cd hallo
!pip install -q -e .
%cd ..

## 3. Download Automatizado dos Modelos Pré-treinados (Checkpoints)

In [ ]:
# Baixar checkpoints do Hallo diretamente do Hugging Face Hub de forma otimizada
from huggingface_hub import snapshot_download
import os

print("📥 Baixando modelos do Hallo (fudan-generative-vision/hallo)...")
target_dir = "./hallo/pretrained_models"
os.makedirs(target_dir, exist_ok=True)

snapshot_download(
    repo_id="fudan-generative-vision/hallo",
    local_dir=target_dir,
    local_dir_use_symlinks=False
)
print("✅ Modelos do Hallo baixados com sucesso!")

## 4. Inicialização do Servidor de Inferência (Gradio WebUI com Túnel Público)

In [ ]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
import gradio as gr
from transformers import AutoModelForImageSegmentation
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Executando no dispositivo: {device.upper()}")

# -----------------------------------------------------
# Carregar Modelo BiRefNet (Segmentação de Cabelo/Fundo)
# -----------------------------------------------------
print("📥 Carregando BiRefNet para matting ultra-realista...")
birefnet_model = AutoModelForImageSegmentation.from_pretrained(
    "ZhengPeng7/BiRefNet",
    trust_remote_code=True
).to(device)
birefnet_model.eval()

transform_image = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def remove_background(image_path):
    image = Image.open(image_path).convert("RGB")
    w, h = image.size
    tensor_in = transform_image(image).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = birefnet_model(tensor_in)[-1].sigmoid().cpu()
    pred = preds[0].squeeze()
    pred_pil = transforms.ToPILImage()(pred).resize((w, h))
    
    image.putalpha(pred_pil)
    output_path = "/content/cutout.png"
    image.save(output_path, "PNG")
    return output_path

# -----------------------------------------------------
# Função de Inferência Hallo (Avatar Animado)
# -----------------------------------------------------
import sys
sys.path.append("./hallo")
from scripts.inference import inference_process
import argparse

def generate_avatar(image_path, audio_path, pose_weight=1.0, face_weight=1.0, lip_weight=1.0, face_expand=1.2):
    config = {
        'source_image': image_path,
        'driving_audio': audio_path,
        'pose_weight': float(pose_weight),
        'face_weight': float(face_weight),
        'lip_weight': float(lip_weight),
        'face_expand_ratio': float(face_expand),
        'config': './hallo/configs/inference/default.yaml',
        'checkpoint': None,
        'output': "/content/avatar_output.mp4"
    }
    args = argparse.Namespace()
    for k, v in config.items():
        setattr(args, k, v)
    
    inference_process(args)
    return "/content/avatar_output.mp4"

# -----------------------------------------------------
# Interface Web Gradio Unificada
# -----------------------------------------------------
with gr.Blocks(title="AI Clone Studio — Aías Fernando") as demo:
    gr.Markdown("# 🎙️ AI Clone Studio: Halo + BiRefNet")
    gr.Markdown("Geração de clones digitais ultra-realistas com micro-expressões sincronizadas e remoção precisa de fundo.")
    
    with gr.Tab("1. Gerar Vídeo do Clone (Halo)"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(label="Foto de Rosto (Alta Resolução)", type="filepath")
                audio_input = gr.Audio(label="Voz Clonada / Áudio de Fala (.wav)", type="filepath")
                with gr.Accordion("Ajustes Finos de Expressão", open=False):
                    pose_w = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="Movimento de Cabeça/Ombros (Pose Weight)")
                    face_w = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="Micro-expressões Faciais (Face Weight)")
                    lip_w = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="Precisão Labial (Lip Weight)")
                    expand = gr.Slider(1.0, 1.5, value=1.2, step=0.05, label="Margem do Enquadramento Facial")
                btn_generate = gr.Button("🎬 Renderizar Vídeo do Clone", variant="primary")
            with gr.Column():
                video_output = gr.Video(label="Vídeo do Clone Gerado")
        
        btn_generate.click(
            fn=generate_avatar,
            inputs=[img_input, audio_input, pose_w, face_w, lip_w, expand],
            outputs=video_output
        )
        
    with gr.Tab("2. Recorte de Fundo Ultra-Preciso (BiRefNet)"):
        with gr.Row():
            with gr.Column():
                cutout_input = gr.Image(label="Imagem para Isolar Fundo", type="filepath")
                btn_cutout = gr.Button("✂️ Isolar com BiRefNet", variant="primary")
            with gr.Column():
                cutout_output = gr.Image(label="Recorte com Fundo Transparente (PNG)")
                
        btn_cutout.click(
            fn=remove_background,
            inputs=cutout_input,
            outputs=cutout_output
        )

# Lançar túnel público gratuito
demo.launch(share=True, debug=True)